## Import libraries


In [1]:
import numpy as np  
import pandas as pd  
import matplotlib.pyplot as plt  
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing import StandardScaler  
from sklearn.tree import DecisionTreeClassifier, plot_tree  
from sklearn.neighbors import KNeighborsClassifier  
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier 
from collections import Counter  
from sklearn.preprocessing import LabelEncoder, MinMaxScaler 
from sklearn.metrics import accuracy_score  
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV






In [2]:
## KNN Implementation


In [3]:
class KNNClassifier:
    def __init__(self, k):
        self.k = k  
        self.X_train = None  
        self.y_train = None  
    def fit(self, X, y):
        self.X_train = X  
        self.y_train = y

    def predict(self, X):  
         
        predictions = [self._predict_single_point(x) for x in X]  
        return np.array(predictions)  

    def _predict_single_point(self, x):  
        
        distance = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))  
        
        knn_indice = np.argsort(distance)[:self.k]  
        
        knn_label = [self.y_train[i] for i in knn_indice]  
        
        most_common = Counter(knn_label).most_common(1)  
        return most_common[0][0]  


## Load and preprocess

#### TO DO:

###### 1-Load data

###### 2-Sample equal number of samples from each class. find a reasonable number.

###### 3-Encode categorical values

###### 4-Keep these columns and drop the rest : grade, term, home_ownership, emp_length

###### 5-Split data to train, validation and test set

###### 6-Scale the data(normalization)

###### 7-The target column is "bad_loans"


In [4]:

def load_and_preprocess_data(path):
    loan_sub = pd.read_csv(path)

    search_parametrs = ['grade', 'term', 'home_ownership', 'emp_length', 'bad_loans']
    loan_sub = loan_sub[search_parametrs]

    loan_sub = loan_sub.dropna(subset=['bad_loans'])

    for col in ['grade', 'term', 'home_ownership', 'emp_length']:
        loan_sub[col] = LabelEncoder().fit_transform(loan_sub[col])

    X = loan_sub.drop('bad_loans', axis=1)
    y = loan_sub['bad_loans']

    train_x, temp_x, train_y, temp_y = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    value_x, test_x, value_y, test_y = train_test_split(temp_x, temp_y, test_size=0.5, random_state=42, stratify=temp_y)

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(train_x)
    X_val_scaled = scaler.transform(value_x)
    X_test_scaled = scaler.transform(test_x)

    smote = SMOTE(random_state=42)
    X_train_scaled, train_y = smote.fit_resample(X_train_scaled, train_y)

    return X_train_scaled, X_val_scaled, X_test_scaled, train_y, value_y, test_y, X.columns


#### Training functions


In [5]:
def train_decision_tree(X_train, y_train, d):  
    model = DecisionTreeClassifier(max_depth=d, random_state=42)  
    model.fit(X_train, y_train)  
    return model  
 vbb
def train_adaboost(X_train, y_train, n):  
    model = AdaBoostClassifier(n_estimators=n)  
    model.fit(X_train, y_train)  
    return model   

def train_rf(X_train, y_train):  
    param_grid = {
    'n_estimators': [50, 100], 
    'max_depth': [10, 20],    
    'min_samples_split': [2],  
    'max_features': ['sqrt', 'log2'], 
    'bootstrap': [True]    
}
    rf_model = RandomForestClassifier(random_state=42)
    
    grid_search = GridSearchCV(rf_model, param_grid, cv=3, n_jobs=-1, scoring='accuracy')
    
    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_

In [6]:
def compare_models(dt_accuracy, knn_accuracy, ab_accuracy, rf_accuracy):
    models = ['Decision Tree', 'KNN','Adaboost', 'Random Forest']
    accuracies = [dt_accuracy, knn_accuracy, ab_accuracy,rf_accuracy]

    plt.figure(figsize=(8, 6))
    plt.bar(models, accuracies)
    plt.title('Model Comparison - Test Accuracy')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1)
    for i, v in enumerate(accuracies):
        plt.text(i, v + 0.01, f'{v:.4f}', ha='center')
    plt.show()

## The main function

#### TO DO:

###### Use the defined functions to load the dataset and train the models.

###### You should optimize the hyperparameters. Maximum depth for DT and k for KNN and n_estimators for Adaboost.

###### Find the best DT, KNN, Adaboost and RF model and report the test accuracy.


In [ ]:

def main():
    path = r"D:\UNI\Artificial Intelligence\Project\loan_sub.csv"  
    X_train, X_val, X_test, y_train, y_val, y_test, columns = load_and_preprocess_data(path)  

    best_dt_model = None  
    best_dt_acc = 0  
    for depth in range(1, 11):  
        dt_model = train_decision_tree(X_train, y_train, depth)  
        val_pred = dt_model.predict(X_val)  
        val_acc = accuracy_score(y_val, val_pred)  
        if val_acc > best_dt_acc:  
            best_dt_acc = val_acc  
            best_dt_model = dt_model  

    best_knn_model = None  
    best_knn_acc = 0  
    for k in range(1, 21): 
        knn_model = train_knn(X_train, y_train, k=k)  
        val_pred = knn_model.predict(X_val)  
        val_acc = accuracy_score(y_val, val_pred)  
        if val_acc > best_knn_acc:  
            best_knn_acc = val_acc  
            best_knn_model = knn_model  

    best_ab_model = None  
    best_ab_acc = 0  
    for n in range(10, 110, 10):  
        ab_model = train_adaboost(X_train, y_train,n)  
        val_pred = ab_model.predict(X_val)  
        val_acc = accuracy_score(y_val, val_pred)  
        if val_acc > best_ab_acc:  
            best_ab_acc = val_acc  
            best_ab_model = ab_model  

    best_rf_model = None  
    best_rf_acc = 0  
    rf_model = train_rf(X_train, y_train)  
    val_pred = rf_model.predict(X_val)  
    val_acc = accuracy_score(y_val, val_pred)
    best_rf_acc = val_acc
    best_rf_model = rf_model

    dt_test_acc = accuracy_score(y_test, best_dt_model.predict(X_test))  
    knn_test_acc = accuracy_score(y_test, best_knn_model.predict(X_test))  
    ab_test_acc = accuracy_score(y_test, best_ab_model.predict(X_test))  
    rf_test_acc = accuracy_score(y_test, best_rf_model.predict(X_test))   

    print(f"DT  Accuracy: {dt_test_acc:.4f}")  
    print(f"KNN  Accuracy: {knn_test_acc:.4f}")  
    print(f"AdaBoost  Accuracy: {ab_test_acc:.4f}")  
    print(f"RF  Accuracy: {rf_test_acc:.4f}")        
    compare_models(dt_test_acc, knn_test_acc, ab_test_acc, rf_test_acc)

    plt.figure(figsize=(60,36))
    plot_tree(best_dt_model, 
            feature_names=columns, 
            class_names=["0", "1"],       
            filled=True,
            rounded=True,
            fontsize=8
            )
    plt.show()


if __name__ == "__main__":
    main()

C:\Users\LOQ\AppData\Local\Temp\ipykernel_22764\1882068033.py:2: DtypeWarning: Columns (19,47) have mixed types. Specify dtype option on import or set low_memory=False.
  loan_sub = pd.read_csv(path)
